# FingerCount: A Neural Network Approach to Gesture Recognition

In [ ]:
#pip install tensorflow
#pip install tensorflow.keras

In [2]:
import os
import cv2 as cv
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models
from sklearn.model_selection import train_test_split

## Train CNN

In [ ]:

# 1. Update Project Constants for 128x128 Resolution
DATASET_PATH = "../../data/hands"   
IMG_SIZE = 128                # Changed to 128 to match your dataset exactly
CLASSES = ['0', '1', '2', '3', '4', '5']

images = []
labels = []

print("[INFO] Loading 128x128 black-and-white Kaggle images...")

# 2. Load the categorized images
for label_idx, class_name in enumerate(CLASSES):
    class_dir = os.path.join(DATASET_PATH, class_name)
    if not os.path.exists(class_dir):
        print(f"[WARNING] Folder {class_dir} missing. Skipping.")
        continue
        
    for img_name in os.listdir(class_dir):
        try:
            img_path = os.path.join(class_dir, img_name)
            # Read explicitly in grayscale
            img = cv.imread(img_path, cv.IMREAD_GRAYSCALE)
            if img is None:
                continue
            
            # Ensure it is exactly 128x128
            img = cv.resize(img, (IMG_SIZE, IMG_SIZE))
            images.append(img)
            labels.append(label_idx)
        except Exception:
            pass

# Normalize pixel values from [0-255] to [0.0-1.0]
X = np.array(images, dtype="float32") / 255.0
y = np.array(labels, dtype="int32")

# Reshape to (Batch Size, 128, 128, 1) for grayscale CNN input
X = X.reshape(-1, IMG_SIZE, IMG_SIZE, 1)

# Split into 80% Training and 20% Validation
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)
print(f"[INFO] Data loaded. Training size: {len(X_train)} | Validation size: {len(X_val)}")

# 3. Optimized CNN Architecture for 128x128 Images
model = models.Sequential([
    # Input layer matched to 128x128x1
    layers.Conv2D(32, (3, 3), activation='relu', input_shape=(IMG_SIZE, IMG_SIZE, 1)),
    layers.MaxPooling2D((2, 2)),
    
    layers.Conv2D(64, (3, 3), activation='relu'),
    layers.MaxPooling2D((2, 2)),
    
    layers.Conv2D(128, (3, 3), activation='relu'), # Added an extra layer for higher detail handling
    layers.MaxPooling2D((2, 2)),
    
    layers.Flatten(),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.5), # Regularization to prevent overfitting
    layers.Dense(6, activation='softmax') # 6 nodes corresponding to classes 0 through 5
])

# 4. Compile the Model
model.compile(optimizer='adam',
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])

# 5. Train the Model
print("[INFO] Training started...")
model.fit(X_train, y_train, epochs=10, batch_size=32, validation_data=(X_val, y_val))

# 6. Save Model
model.save('hand_gesture_model.h5')
print("[SUCCESS] Model trained and saved as 'hand_gesture_model.h5'")


[INFO] Loading 128x128 black-and-white Kaggle images...
[INFO] Data loaded. Training size: 9604 | Validation size: 2402


c:\Users\QC\AppData\Local\Programs\Python\Python310\lib\site-packages\keras\src\layers\convolutional\base_conv.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


[INFO] Training started...
Epoch 1/10
301/301 ━━━━━━━━━━━━━━━━━━━━ 371s 1s/step - accuracy: 0.9019 - loss: 0.2756 - val_accuracy: 0.9996 - val_loss: 0.0012
Epoch 2/10
301/301 ━━━━━━━━━━━━━━━━━━━━ 385s 1s/step - accuracy: 0.9959 - loss: 0.0139 - val_accuracy: 1.0000 - val_loss: 1.6379e-04
Epoch 3/10
301/301 ━━━━━━━━━━━━━━━━━━━━ 614s 2s/step - accuracy: 0.9966 - loss: 0.0085 - val_accuracy: 1.0000 - val_loss: 1.2355e-05
Epoch 4/10
301/301 ━━━━━━━━━━━━━━━━━━━━ 184s 611ms/step - accuracy: 0.9990 - loss: 0.0034 - val_accuracy: 1.0000 - val_loss: 5.9341e-05
Epoch 5/10
301/301 ━━━━━━━━━━━━━━━━━━━━ 195s 648ms/step - accuracy: 0.9979 - loss: 0.0062 - val_accuracy: 1.0000 - val_loss: 3.7271e-06
Epoch 6/10
301/301 ━━━━━━━━━━━━━━━━━━━━ 182s 605ms/step - accuracy: 0.9997 - loss: 0.0011 - val_accuracy: 1.0000 - val_loss: 5.2621e-07
Epoch 7/10
301/301 ━━━━━━━━━━━━━━━━━━━━ 171s 569ms/step - accuracy: 0.9986 - loss: 0.0048 - val_accuracy: 1.0000 - val_loss: 8.6547e-07
Epoch 8/10
301/301 ━━━━━━━━━━━━━━━

[SUCCESS] Model trained and saved as 'hand_gesture_model.h5'


## Prediction

In [44]:
import cv2 as cv
import numpy as np
import tensorflow as tf

# 1. Load and Compile Model
try:
    model = tf.keras.models.load_model('hand_gesture_model.h5')
    model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    print("[SUCCESS] Loaded and compiled model.")
except Exception as e:
    raise FileNotFoundError("Could not find 'hand_gesture_model.h5'")

# 2. Load Test Image
image_target_path = '../../data/hands/1/1_1.png' 
hand_image = cv.imread(image_target_path)

if hand_image is None:
    raise FileNotFoundError(f"Image not found at: {image_target_path}")

# Configuration matching 128x128 Kaggle dataset structure
threshold_value = 150
IMG_SIZE = 128 
CLASSES = ['0 Fingers (Fist)', '1 Finger', '2 Fingers', '3 Fingers', '4 Fingers', '5 Fingers']

# Configure Windows to be Large and Resizable
cv.namedWindow('Original Image - CNN Classification', cv.WINDOW_NORMAL)
cv.namedWindow('Threshold Mask Fed To CNN', cv.WINDOW_NORMAL)

cv.resizeWindow('Original Image - CNN Classification', 800, 600)
cv.resizeWindow('Threshold Mask Fed To CNN', 500, 500)

print("\n--- Controls ---")
print("Press 'w' to INCREASE threshold")
print("Press 's' to DECREASE threshold")
print("Press 'q' to QUIT\n")

while True:
    # reset from the pure, clean original image on each loop iteration to avoid compounding text overlays and thresholding artifacts
    frame = hand_image.copy()
    
    # process the image before overlaying any text, ensuring the CNN receives a pristine mask for inference
    gray = cv.cvtColor(frame, cv.COLOR_BGR2GRAY)
    blur = cv.GaussianBlur(gray, (9, 9), 0)
    
    # Swapped to THRESH_BINARY so the hand is white and background is black
    _, thresh = cv.threshold(blur, threshold_value, 255, cv.THRESH_BINARY)

    # Initialize variables for drawing
    display_text = "CNN: Predicting..."

    try:
        # 3. Prepare the mask image for the CNN
        cnn_input = cv.resize(thresh, (IMG_SIZE, IMG_SIZE)) 
        cnn_input = cnn_input.astype("float32") / 255.0     
        
        # Reshape to a 4D tensor matrix: (1, 128, 128, 1)
        cnn_input = np.expand_dims(cnn_input, axis=-1)
        cnn_input = np.expand_dims(cnn_input, axis=0)
        
        # 4. Predict
        predictions = model.predict(cnn_input, verbose=0)
        class_id = np.argmax(predictions[0])
        confidence = predictions[0][class_id] * 100
        
        display_text = f"CNN: {CLASSES[class_id]} ({confidence:.1f}%)"
        
    except Exception as e:
        print(f"[ERROR] Inference failure: {e}")
            
    # Display predicted class and threshold on the original image
    cv.putText(frame, display_text, (10, 10), cv.FONT_HERSHEY_SIMPLEX, 0.3, (0, 255, 0), 1)
    cv.putText(frame, f"Threshold: {threshold_value}", (30, frame.shape[0] - 40), 
               cv.FONT_HERSHEY_SIMPLEX, 0.3, (0, 0, 255), 1)
    
    # Display the final outputs
    cv.imshow('Original Image - CNN Classification', frame)
    cv.imshow('Threshold Mask Fed To CNN', thresh)
    
    key = cv.waitKey(30) & 0xFF
    if key == ord('q'):
        break
    elif key == ord('w'): 
        threshold_value = min(255, threshold_value + 5)
    elif key == ord('s'): 
        threshold_value = max(0, threshold_value - 5)

cv.destroyAllWindows()


[SUCCESS] Loaded and compiled model.

--- Controls ---
Press 'w' to INCREASE threshold
Press 's' to DECREASE threshold
Press 'q' to QUIT

